# Getting the laws' discussion

With this notebook we extract the "discusion en sala" from each law

In [7]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import time
import re
from datetime import datetime, date

We start setting up the requeriments to do a proper webscrapping

In [11]:
import requests
from pathlib import Path
import re

endpoint = "https://datos.bcn.cl/sparql"
section_uri = "http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-po1-ds61"

query = f"""
PREFIX bcnres: <http://datos.bcn.cl/ontologies/bcn-resources#>
SELECT ?xml WHERE {{
  <{section_uri}> bcnres:descripcionXML ?xml .
}}
LIMIT 1
"""

r = requests.get(
    endpoint,
    params={"query": query, "format": "application/sparql-results+json"},
    headers={"Accept": "application/sparql-results+json", "User-Agent": "Mozilla/5.0"},
    timeout=30,
)
r.raise_for_status()
data = r.json()

rows = data["results"]["bindings"]
print("rows:", len(rows))

if not rows:
    print("No descripcionXML found for this section.")
else:
    xml_text = rows[0]["xml"]["value"]
    print("xml preview:", xml_text[:200])

    # VERY simple tag strip (good enough to start; you can improve later)
    plain = re.sub(r"<[^>]+>", "", xml_text)
    plain = re.sub(r"\s+\n", "\n", plain).strip()

    outdir = Path("datos_dump")
    outdir.mkdir(exist_ok=True)
    (outdir / "section_text.txt").write_text(plain, encoding="utf-8")
    print("Saved:", outdir / "section_text.txt")

rows: 1
xml preview:  <debateSection xmlns:bcn="http://datos.bcn.cl" bcn:uriProyectoLey="http://datos.bcn.cl/recurso/cl/proyecto-de-ley/16371-24" bcn:uriResultadoDebate="http://datos.bcn.cl/ontologies/bcn-resources#seApru
Saved: datos_dump\section_text.txt


In [13]:
from pathlib import Path
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

def unescape_js_string(s: str) -> str:
    # enough for BCN’s payload
    s = s.replace("\\\\", "\\")
    s = s.replace("\\'", "'")
    s = s.replace('\\"', '"')
    s = s.replace("\\n", "\n").replace("\\t", "\t").replace("\\r", "\r")
    return s

xml_path = Path("datos_dump/ley_21788_1-4.xml")  # your saved XHR response
xml_text = xml_path.read_text(encoding="utf-8")

root = ET.fromstring(xml_text)
cmd_nodes = root.findall(".//cmd[@n='js']")

target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if "div class=\"item\"" in txt and "$('.modal-body').html(" in txt:
        target_js = txt
        break

if target_js is None:
    raise ValueError("Could not find injected transcript HTML in the XAJAX XML.")

start_token = "$('.modal-body').html('"
start = target_js.find(start_token) + len(start_token)
end = target_js.rfind("');")
raw_html = target_js[start:end]

modal_html = unescape_js_string(raw_html)
soup = BeautifulSoup(modal_html, "html.parser")
item = soup.select_one("div.item")

section_uri = item.get("parteDocumento")   # <-- THIS is what you need
doc_uri = item.get("uriDocumento")

print("section_uri:", section_uri)
print("doc_uri:", doc_uri)

ParseError: no element found: line 1, column 0 (<string>)

AHORA SI

In [1]:
# Chunk 1 — imports and configuration

import re
import json
import time
import random
from pathlib import Path
from datetime import datetime, date
import xml.etree.ElementTree as ET

import requests
import pandas as pd
from bs4 import BeautifulSoup


# -----------------------------
# Project configuration
# -----------------------------

LAW_URLS = [
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/",  # Ley 21.472
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/",  # Ley 21.788
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8393/",  # Ley 21.755
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8443/",  # Ley 21.778
]

START_DATE = date(2022, 3, 11)
END_DATE   = date(2026, 3, 10)

BASE_DIR = Path("results_bcn")
BASE_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "discussion_summary.csv"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

REQUEST_TIMEOUT = 30

# Safety settings
SKIP_IF_ALREADY_SAVED = True
SAVE_RAW_HTML = True
SAVE_RAW_XAJAX_XML = False   # optional; keep False unless you need debugging
PAUSE_BETWEEN_POSTS = (2.5, 5.0)   # random pause between transcript requests
PAUSE_BETWEEN_LAWS = (6.0, 10.0)   # random pause between laws

print("Configured.")
print("Date window:", START_DATE, "to", END_DATE)
print("Output folder:", BASE_DIR.resolve())

Configured.
Date window: 2022-03-11 to 2026-03-10
Output folder: C:\Users\jorge\Documents\Mis documentos\Uni Konstanz\1. ICSS + Python\Project\chilean-polispeech-nlp\notebooks\intake\results_bcn


In [2]:
# Chunk 2 — helper functions

def normalize_space(text):
    """Collapse repeated whitespace and replace non-breaking spaces."""
    if text is None:
        return ""
    text = text.replace("\xa0", " ")
    return " ".join(text.split())


def law_id_from_url(law_url):
    """Extract BCN law id from the URL, e.g. 8393 from .../historia-de-la-ley/8393/"""
    m = re.search(r"/historia-de-la-ley/(\d+)/?", law_url)
    return m.group(1) if m else None


def safe_law_number(law_number, fallback="unknown_law"):
    """Convert a law number like 21.755 into a folder-safe version like 21_755."""
    if not law_number:
        return fallback
    return law_number.replace(".", "_")


def make_session():
    """Create a requests session with a browser-like user agent."""
    s = requests.Session()
    s.headers.update({
        "User-Agent": USER_AGENT,
        "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    })
    return s


def pause_random(seconds_range):
    """Sleep for a random number of seconds inside the given range tuple."""
    low, high = seconds_range
    delay = random.uniform(low, high)
    print(f"Sleeping {delay:.1f}s...")
    time.sleep(delay)

In [3]:
# Chunk 3 — fetch page and save raw HTML

def fetch_law_page(session, law_url, timeout=REQUEST_TIMEOUT):
    """
    Fetch the law page once.
    This GET is important because it also helps establish the session/cookies
    before the AJAX POST requests.
    """
    response = session.get(law_url, timeout=timeout)
    response.raise_for_status()
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    return soup, html


def parse_law_metadata(soup, law_url):
    """
    Extract basic metadata:
    - BCN law id
    - law number (e.g. 21.755)
    - title
    """
    page_text = soup.get_text("\n", strip=True)
    lines = [normalize_space(x) for x in page_text.split("\n") if normalize_space(x)]

    # Law number
    m = re.search(r"Historia de la Ley N°\s*([0-9.]+)", page_text)
    law_number = m.group(1) if m else None

    # Title: usually the next meaningful line after "Historia de la Ley N° ..."
    law_title = None
    if law_number:
        target = f"Historia de la Ley N° {law_number}"
        for i, line in enumerate(lines):
            if target in line:
                for candidate in lines[i+1:i+12]:
                    if candidate.startswith("Publicada:"):
                        continue
                    if "Pantalla completa" in candidate:
                        continue
                    if "Fabrica tu dossier" in candidate:
                        continue
                    if len(candidate) >= 10:
                        law_title = candidate
                        break
                break

    return {
        "law_id": law_id_from_url(law_url),
        "law_number": law_number,
        "law_title": law_title,
        "history_url": law_url,
    }


def get_law_folder(base_dir, law_meta):
    """
    Create one folder per law.
    """
    folder_name = f"law_{safe_law_number(law_meta.get('law_number'))}"
    law_folder = base_dir / folder_name
    law_folder.mkdir(parents=True, exist_ok=True)
    return law_folder

In [4]:
# Chunk 4 — discover candidate discussion entries from the Índice

def discover_discussions_from_index(soup):
    """
    Parse the Índice tree (#arbol) and collect every 'Discusión en Sala' entry,
    together with the stage it belongs to.
    """
    arbol = soup.select_one("#arbol")
    if arbol is None:
        raise ValueError("Could not find #arbol (Índice) in the page HTML.")

    anchor_texts = [
        normalize_space(a.get_text(" ", strip=True))
        for a in arbol.select("a")
    ]
    anchor_texts = [x for x in anchor_texts if x]

    stage_re = re.compile(r"^(\d+)\.\s+(.*)$")
    item_re = re.compile(r"^(\d+)\.(\d+)\.\s+(.*?)\s*\((\d{2}-\d{2}-\d{4})\)$")

    rows = []

    current_stage_num = None
    current_stage_text = None

    for text in anchor_texts:
        # Example: "1. Primer Trámite Constitucional: Cámara de Diputados"
        m_stage = stage_re.match(text)
        if m_stage and "Trámite" in text:
            current_stage_num = int(m_stage.group(1))
            current_stage_text = m_stage.group(2)
            continue

        # Example: "1.5. Discusión en Sala (05-03-2025)"
        m_item = item_re.match(text)
        if m_item and current_stage_num is not None:
            stage_num = int(m_item.group(1))
            item_num = int(m_item.group(2))
            label = m_item.group(3)
            date_str = m_item.group(4)

            # Safety check: the item should belong to the current stage
            if stage_num != current_stage_num:
                continue

            if "discusión en sala" in label.lower():
                event_date = datetime.strptime(date_str, "%d-%m-%Y").date()
                stage_text_low = current_stage_text.lower()

                rows.append({
                    "stage_num": stage_num,
                    "stage_text": current_stage_text,
                    "stage_text_lower": stage_text_low,
                    "section_num": f"{stage_num}.{item_num}",
                    "section_key": f"{stage_num}-{item_num}",
                    "label": label,
                    "event_date": event_date,
                    "stage_is_constitutional": "trámite constitucional" in stage_text_low,
                    "stage_has_diputados": "cámara de diputados" in stage_text_low,
                    "stage_has_senado": "senado" in stage_text_low,
                    "stage_is_mixed": (
                        "diputados-senado" in stage_text_low
                        or ("cámara de diputados" in stage_text_low and "senado" in stage_text_low)
                    ),
                    "stage_is_finalization": "finalización" in stage_text_low,
                })

    return pd.DataFrame(rows)

In [5]:
# Chunk 5 — filter to target discussions

def filter_target_discussions(df, start_date=START_DATE, end_date=END_DATE):
    """
    Keep only discussions that:
    - are inside the target period
    - are in Trámite Constitucional
    - are in Cámara de Diputados only
    - are not mixed Cámara+Senado
    - are not Trámite Finalización
    """
    if df.empty:
        return df.copy()

    mask = (
        (df["event_date"] >= start_date) &
        (df["event_date"] <= end_date) &
        (df["stage_is_constitutional"]) &
        (df["stage_has_diputados"]) &
        (~df["stage_has_senado"]) &
        (~df["stage_is_mixed"]) &
        (~df["stage_is_finalization"])
    )

    return df.loc[mask].copy()

In [6]:
# Chunk 6 — fetch transcript through AJAX

def extract_modal_html_from_xajax(response_text):
    """
    The xajax response is XML containing several JS commands.
    We want the JS command that injects the transcript HTML into $('.modal-body').
    """
    root = ET.fromstring(response_text)
    cmd_nodes = root.findall(".//cmd[@n='js']")

    candidates = []
    for node in cmd_nodes:
        txt = node.text or ""
        if "$('.modal-body').html(" in txt:
            candidates.append(txt)

    if not candidates:
        raise ValueError("No modal-body injection found in xajax response.")

    # Prefer the longest candidate containing div.item
    target_js = None
    for txt in sorted(candidates, key=len, reverse=True):
        if 'div class="item"' in txt:
            target_js = txt
            break

    if target_js is None:
        target_js = max(candidates, key=len)

    # Try single-quoted payload first
    start_token = "$('.modal-body').html('"
    end_token = "');"

    if start_token in target_js:
        start = target_js.find(start_token) + len(start_token)
        end = target_js.rfind(end_token)
        raw_html = target_js[start:end]
    else:
        # Fallback for double quotes if needed
        start_token = "$('.modal-body').html(\""
        end_token = "\");"
        start = target_js.find(start_token)
        if start == -1:
            raise ValueError("Could not locate modal-body HTML start token.")
        start += len(start_token)
        end = target_js.rfind(end_token)
        raw_html = target_js[start:end]

    # Unescape JS string escapes
    modal_html = raw_html
    modal_html = modal_html.replace("\\\\", "\\")
    modal_html = modal_html.replace("\\'", "'")
    modal_html = modal_html.replace('\\"', '"')
    modal_html = modal_html.replace("\\n", "\n")
    modal_html = modal_html.replace("\\t", "\t")
    modal_html = modal_html.replace("\\r", "\r")

    return modal_html


def parse_modal_html(modal_html):
    """
    Extract transcript text and useful metadata from the modal HTML.
    """
    soup = BeautifulSoup(modal_html, "html.parser")

    item = soup.select_one("div.item")
    container = item if item is not None else soup

    attrs = dict(item.attrs) if item else {}

    paragraphs = [
        normalize_space(p.get_text(" ", strip=True))
        for p in container.find_all("p")
    ]
    paragraphs = [p for p in paragraphs if p]

    if paragraphs:
        transcript_text = "\n\n".join(paragraphs)
    else:
        transcript_text = normalize_space(container.get_text("\n", strip=True))

    upper_text = transcript_text.upper()

    preliminary_markers = [
        "LA VERSIÓN OFICIAL DE ESTA DISCUSIÓN EN SALA NO SE ENCUENTRA DISPONIBLE",
        "VERSIÓN PRELIMINAR",
    ]

    is_preliminary = any(marker in upper_text for marker in preliminary_markers)

    return {
        "item_attrs": attrs,
        "paragraphs": paragraphs,
        "paragraph_count": len(paragraphs),
        "transcript_text": transcript_text,
        "char_count": len(transcript_text),
        "is_preliminary": is_preliminary,
    }


def fetch_discussion_transcript(session, law_url, section_key, timeout=REQUEST_TIMEOUT, retry_once=True):
    """
    Retrieve the full transcript for one section_key like '1-5' or '3-1'.
    This uses the AJAX route your first code already proved works.
    """
    headers = {
        "User-Agent": USER_AGENT,
        "Referer": law_url,
        "Origin": "https://www.bcn.cl",
    }

    payload = [
        ("xajax", "verTextoCompleto"),
        ("xajaxr", str(int(time.time() * 1000))),
        ("xajaxargs[]", section_key),
    ]

    response = session.post(law_url, headers=headers, data=payload, timeout=timeout)

    # Optional one-time refresh if 405 appears
    if response.status_code == 405 and retry_once:
        print(f"405 received for {section_key}. Trying one refresh GET and one retry...")
        pause_random((4.0, 7.0))
        session.get(law_url, timeout=timeout)
        pause_random((3.0, 5.0))

        payload = [
            ("xajax", "verTextoCompleto"),
            ("xajaxr", str(int(time.time() * 1000))),
            ("xajaxargs[]", section_key),
        ]
        response = session.post(law_url, headers=headers, data=payload, timeout=timeout)

    response.raise_for_status()

    modal_html = extract_modal_html_from_xajax(response.text)
    parsed = parse_modal_html(modal_html)

    return {
        "section_key": section_key,
        "raw_xajax_xml": response.text,
        "modal_html": modal_html,
        **parsed,
    }

In [7]:
# Chunk 7 — save outputs

def transcript_paths(law_folder, event_date, section_key):
    """
    Build output paths for one discussion.
    """
    stamp = event_date.isoformat()
    key = section_key.replace("-", "_")

    txt_path = law_folder / f"discussion_{stamp}_{key}.txt"
    json_path = law_folder / f"discussion_{stamp}_{key}.json"
    xml_path = law_folder / f"discussion_{stamp}_{key}_raw_xajax.xml"

    return txt_path, json_path, xml_path


def save_transcript_outputs(law_folder, law_meta, entry, transcript_payload, save_raw_xajax=False):
    """
    Save transcript text and metadata.
    """
    txt_path, json_path, xml_path = transcript_paths(
        law_folder=law_folder,
        event_date=entry["event_date"],
        section_key=entry["section_key"],
    )

    # Save text
    txt_path.write_text(transcript_payload["transcript_text"], encoding="utf-8")

    # Save JSON
    bundle = {
        "law_meta": law_meta,
        "entry": {
            "stage_num": entry["stage_num"],
            "stage_text": entry["stage_text"],
            "section_num": entry["section_num"],
            "section_key": entry["section_key"],
            "label": entry["label"],
            "event_date": entry["event_date"].isoformat(),
        },
        "item_attrs": transcript_payload["item_attrs"],
        "paragraph_count": transcript_payload["paragraph_count"],
        "char_count": transcript_payload["char_count"],
        "is_preliminary": transcript_payload["is_preliminary"],
        "transcript_text": transcript_payload["transcript_text"],
    }

    json_path.write_text(
        json.dumps(bundle, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    # Optional raw XML
    if save_raw_xajax:
        xml_path.write_text(transcript_payload["raw_xajax_xml"], encoding="utf-8")

    return txt_path, json_path, xml_path


def load_saved_json(json_path):
    """
    Load previously saved JSON so the notebook can skip repeat requests.
    """
    return json.loads(json_path.read_text(encoding="utf-8"))


def update_summary_csv(summary_csv_path, rows):
    """
    Create or update the global summary CSV.
    Deduplicate by history_url + section_key.
    """
    new_df = pd.DataFrame(rows)

    if summary_csv_path.exists():
        old_df = pd.read_csv(summary_csv_path)
        full_df = pd.concat([old_df, new_df], ignore_index=True)
        full_df = full_df.drop_duplicates(subset=["history_url", "section_key"], keep="last")
    else:
        full_df = new_df.copy()

    full_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
    return full_df

In [8]:
# Chunk 8 — full pipeline for one law

def run_one_law(law_url):
    print("\n" + "=" * 70)
    print("Processing:", law_url)

    session = make_session()

    # 1) GET the page once
    soup, raw_html = fetch_law_page(session, law_url)
    law_meta = parse_law_metadata(soup, law_url)
    law_folder = get_law_folder(BASE_DIR, law_meta)

    print("Law number:", law_meta["law_number"])
    print("Law title :", law_meta["law_title"])

    if SAVE_RAW_HTML:
        raw_html_path = law_folder / "raw_page.html"
        raw_html_path.write_text(raw_html, encoding="utf-8")

    # 2) Discover all discussion entries
    all_discussions = discover_discussions_from_index(soup)

    print("\nAll 'Discusión en Sala' found in the index:")
    if all_discussions.empty:
        print("None found.")
        return []

    display_cols = ["section_num", "event_date", "stage_text"]
    print(all_discussions[display_cols].to_string(index=False))

    # 3) Filter to the target ones
    target_discussions = filter_target_discussions(all_discussions)

    print("\nTarget discussions after applying your rules:")
    if target_discussions.empty:
        print("No target discussions found.")
        return []

    print(target_discussions[display_cols].to_string(index=False))

    summary_rows = []

    # 4) Fetch each target transcript
    target_records = target_discussions.to_dict(orient="records")

    for i, entry in enumerate(target_records, start=1):
        print("\n" + "-" * 50)
        print(f"Fetching {entry['section_num']} ({entry['section_key']}) | {entry['event_date']}")

        txt_path, json_path, xml_path = transcript_paths(
            law_folder=law_folder,
            event_date=entry["event_date"],
            section_key=entry["section_key"],
        )

        # Skip if already saved
        if SKIP_IF_ALREADY_SAVED and json_path.exists():
            print("Already saved. Skipping new request.")
            saved = load_saved_json(json_path)

            summary_rows.append({
                "law_id": law_meta["law_id"],
                "law_number": law_meta["law_number"],
                "law_title": law_meta["law_title"],
                "history_url": law_meta["history_url"],
                "stage_num": entry["stage_num"],
                "stage_text": entry["stage_text"],
                "section_num": entry["section_num"],
                "section_key": entry["section_key"],
                "event_date": entry["event_date"].isoformat(),
                "label": entry["label"],
                "parteDocumento": saved.get("item_attrs", {}).get("partedocumento"),
                "modal_fecha": saved.get("item_attrs", {}).get("fecha"),
                "paragraph_count": saved.get("paragraph_count"),
                "char_count": saved.get("char_count"),
                "is_preliminary": saved.get("is_preliminary"),
                "keep_for_analysis": not bool(saved.get("is_preliminary")),
                "status": "already_saved",
                "txt_path": str(txt_path),
                "json_path": str(json_path),
            })
            continue

        try:
            transcript_payload = fetch_discussion_transcript(
                session=session,
                law_url=law_url,
                section_key=entry["section_key"],
            )

            txt_path, json_path, xml_path = save_transcript_outputs(
                law_folder=law_folder,
                law_meta=law_meta,
                entry=entry,
                transcript_payload=transcript_payload,
                save_raw_xajax=SAVE_RAW_XAJAX_XML,
            )

            print("Saved TXT :", txt_path.name)
            print("Saved JSON:", json_path.name)
            print("Paragraphs:", transcript_payload["paragraph_count"])
            print("Chars     :", transcript_payload["char_count"])
            print("Preliminary version?", transcript_payload["is_preliminary"])

            summary_rows.append({
                "law_id": law_meta["law_id"],
                "law_number": law_meta["law_number"],
                "law_title": law_meta["law_title"],
                "history_url": law_meta["history_url"],
                "stage_num": entry["stage_num"],
                "stage_text": entry["stage_text"],
                "section_num": entry["section_num"],
                "section_key": entry["section_key"],
                "event_date": entry["event_date"].isoformat(),
                "label": entry["label"],
                "parteDocumento": transcript_payload["item_attrs"].get("partedocumento"),
                "modal_fecha": transcript_payload["item_attrs"].get("fecha"),
                "paragraph_count": transcript_payload["paragraph_count"],
                "char_count": transcript_payload["char_count"],
                "is_preliminary": transcript_payload["is_preliminary"],
                "keep_for_analysis": not transcript_payload["is_preliminary"],
                "status": "retrieved",
                "txt_path": str(txt_path),
                "json_path": str(json_path),
            })

        except Exception as exc:
            print("Failed:", exc)

            summary_rows.append({
                "law_id": law_meta["law_id"],
                "law_number": law_meta["law_number"],
                "law_title": law_meta["law_title"],
                "history_url": law_meta["history_url"],
                "stage_num": entry["stage_num"],
                "stage_text": entry["stage_text"],
                "section_num": entry["section_num"],
                "section_key": entry["section_key"],
                "event_date": entry["event_date"].isoformat(),
                "label": entry["label"],
                "parteDocumento": None,
                "modal_fecha": None,
                "paragraph_count": None,
                "char_count": None,
                "is_preliminary": None,
                "keep_for_analysis": None,
                "status": f"failed: {exc}",
                "txt_path": "",
                "json_path": "",
            })

        # Pause between transcript POSTs
        if i < len(target_records):
            pause_random(PAUSE_BETWEEN_POSTS)

    return summary_rows

In [11]:
# Chunk 9 — test with one law first

test_rows = run_one_law("https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/")

if test_rows:
    summary_df = update_summary_csv(SUMMARY_CSV, test_rows)
    print("\nUpdated summary CSV:")
    print(SUMMARY_CSV)
    display(summary_df.tail(len(test_rows)))


Processing: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/


HTTPError: 405 Client Error: Method Not Allowed for url: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/

In [10]:
# Chunk 10 — run on multiple laws

all_rows = []

for i, law_url in enumerate(LAW_URLS, start=1):
    rows = run_one_law(law_url)
    if rows:
        all_rows.extend(rows)
        update_summary_csv(SUMMARY_CSV, rows)

    if i < len(LAW_URLS):
        pause_random(PAUSE_BETWEEN_LAWS)

print("\nDone.")
print("Summary CSV:", SUMMARY_CSV.resolve())

if SUMMARY_CSV.exists():
    summary_df = pd.read_csv(SUMMARY_CSV)
    display(summary_df)


Processing: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/


HTTPError: 405 Client Error: Method Not Allowed for url: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/